### 🏎️ TP : Base de données sur les voitures 🏎️



In [1]:
.schema

sql
"CREATE TABLE sqlite_sequence(name,seq)"
"CREATE TABLE ""brands"" ( ""brand_id"" INTEGER, ""brand_name"" VARCHAR(50) NOT NULL, PRIMARY KEY(""brand_id"" AUTOINCREMENT) )"
"CREATE TABLE ""parts"" ( ""part_id"" INTEGER, ""part_name"" VARCHAR(100) NOT NULL, ""manufacture_start_date"" DATE NOT NULL, ""manufacture_end_date"" DATE, ""part_recall"" INTEGER DEFAULT 0 CHECK(""part_recall"" = 0 OR ""part_recall"" = 1), PRIMARY KEY(""part_id"" AUTOINCREMENT) )"
"CREATE TABLE ""options"" ( ""option_set_id"" INTEGER, ""model_id"" INTEGER, ""engine_id"" INTEGER NOT NULL, ""transmission_id"" INTEGER NOT NULL, ""chassis_id"" INTEGER NOT NULL, ""premium_sound_id"" INTEGER, ""color"" VARCHAR(30) NOT NULL, ""option_set_price"" INTEGER NOT NULL, FOREIGN KEY(""transmission_id"") REFERENCES ""parts""(""part_id""), FOREIGN KEY(""model_id"") REFERENCES ""models""(""model_id""), FOREIGN KEY(""premium_sound_id"") REFERENCES ""parts""(""part_id""), FOREIGN KEY(""engine_id"") REFERENCES ""parts""(""part_id""), FOREIGN KEY(""chassis_id"") REFERENCES ""parts""(""part_id""), PRIMARY KEY(""option_set_id"" AUTOINCREMENT) )"
"CREATE TABLE ""models"" ( ""model_id"" INTEGER, ""model_name"" VARCHAR(50) NOT NULL, ""model_base_price"" INTEGER NOT NULL, ""brand_id"" INTEGER NOT NULL, FOREIGN KEY(""brand_id"") REFERENCES ""brands""(""brand_id""), PRIMARY KEY(""model_id"" AUTOINCREMENT) )"
"CREATE TABLE ""customers"" ( ""customer_id"" INTEGER, ""first_name"" VARCHAR(50) NOT NULL, ""last_name"" VARCHAR(50) NOT NULL, ""gender"" STRING CHECK(""gender"" = ""Male"" OR ""gender"" = ""Female""), ""household_income"" INTEGER, ""birthdate"" DATE NOT NULL, ""phone_number"" INTEGER NOT NULL, ""email"" VARCHAR(128), PRIMARY KEY(""customer_id"" AUTOINCREMENT) )"
"CREATE TABLE ""ownerships"" ( ""customer_id"" INTEGER NOT NULL, ""vin"" INTEGER NOT NULL, ""purchase_date"" DATE NOT NULL, ""purchase_price"" INTEGER NOT NULL, ""warantee_expire_date"" DATE, FOREIGN KEY(""customer_id"") REFERENCES ""customers""(""customer_id""), FOREIGN KEY(""vin"") REFERENCES ""cars""(""vin""), PRIMARY KEY(""customer_id"",""vin"") )"
"CREATE TABLE ""cars"" ( ""vin"" INTEGER, ""model_id"" INTEGER NOT NULL, ""option_set_id"" INTEGER NOT NULL, ""manufactured_date"" DATE NOT NULL, FOREIGN KEY(""model_id"") REFERENCES ""models""(""model_id""), FOREIGN KEY(""option_set_id"") REFERENCES ""options""(""option_set_id""), PRIMARY KEY(""vin"" AUTOINCREMENT) )"


#### A - Requêtes simples

1. Afficher la liste des marques (`brand_name`) présentes dans la base.

In [2]:
SELECT brand_name FROM brands

brand_name
Tiger
Supreme
Yellow
Ferrari
SuperCar
Bugatti


2. Afficher les noms et prénoms des clients, triés par ordre alphabétique des noms de famille.

_Attention, une même personne peut avoir acheté plusieurs voitures et on ne veut pas de doublons._ 

In [3]:
SELECT DISTINCT last_name, first_name FROM customers
ORDER BY last_name

last_name,first_name
Bellacio,Monica
Bellacio,Peter
DuBois,Morgane
DuLac,Lancelot
Enchanteur,Merlin
Etegal,Sam
Hughes,Jessica
Jacobs,Jeremy
Jacobs,Patrick
Korkova,Maria


#### B - Requêtes avec une jointure

3. Afficher la liste des noms des modèles (`model_name`) de voitures de la marque *SuperCar*.

In [5]:
SELECT model_name FROM brands b, models m
WHERE b.brand_id = m.brand_id
AND brand_name = "SuperCar"

model_name
Gabriela DeLuxe
Antonia DeLuxe


4. Madame Maria Swabota a acheté deux voitures dans sa vie. 

Afficher les dates d'achat (`purchase_date`) et les prix d'achat (`purchase_price`) de ces deux voitures.

In [6]:
SELECT purchase_date, purchase_price FROM ownerships o, customers c
WHERE o.customer_id = c.customer_id
AND last_name = "Swabota"
AND first_name = "Maria"

purchase_date,purchase_price
2015-12-01,121300
2022-05-17,29000


In [7]:
SELECT purchase_date, purchase_price FROM ownerships
WHERE customer_id = (
        SELECT customer_id FROM customers
        WHERE last_name = "Swabota"
        AND first_name = "Maria")

purchase_date,purchase_price
2015-12-01,121300
2022-05-17,29000


5. Certaines voitures de la base ont été fabriquées entre le premier janvier 2020 et maintenant.

Afficher le nom des modèles de ces voitures et leur date de fabication (`manufactured_date`).

_Comme pour les nombres, on peut faire en SQL des comparaisons sur les attributs de type DATE avec des clauses de la forme_ `manufactured_date > '2020-01-01'`.

In [8]:
SELECT model_name, manufactured_date FROM cars c, models m
WHERE m.model_id = c.model_id
AND manufactured_date > '2020-01-01'

model_name,manufactured_date
The Brunette,2020-11-02
The Blonde,2020-11-03
The Red Head,2021-11-02
Z4 Gordini,2022-11-12
Z5 Cordoba,2020-09-09
Z6 Kingaro,2020-11-02
Orange,2022-04-17
LaFerrari,2020-11-02
450,2020-02-07
F12 Berlinetta,2020-07-14


6. Afficher le nombre de voitures rouges qui ont été fabriquées. 

_La couleur est une des `options` d'une voiture fabriquée_.

In [10]:
SELECT DISTINCT color FROM options

color
Blue
Yellow
Green
Red
Cyan
Purple
Black
White
Magenta


In [13]:
/* Écrire votre requête
On doit trouver 5 voitures rouges */
SELECT COUNT(*) AS "Nombre de voitures rouges" FROM cars c, options o
WHERE c.option_set_id = o.option_set_id
AND UPPER(color) = "RED"

Nombre de voitures rouges
5


7. Expliquer ce que fournit la requête suivante :

In [14]:
SELECT gender,SUM(purchase_price)
FROM ownerships
JOIN customers ON customers.customer_id = ownerships.customer_id
GROUP BY gender;

gender,SUM(purchase_price)
Female,804700
Male,686400


#### C - Requêtes avec des jointures multiples

8. Afficher la liste des noms et prénoms des femmes qui ont acheté une voiture bleue, ainsi que la date d'achat (`purchase_date`) de cette voiture bleue.

In [16]:
/* Écrire votre requête
On doit trouver une liste de 4 personnes */
SELECT last_name, first_name, purchase_date
FROM options op, cars ca, ownerships ow, customers cu
WHERE op.option_set_id = ca.option_set_id
AND ca.vin = ow.vin
AND ow.customer_id = cu.customer_id
AND LOWER(color) = "blue"
AND LOWER(gender) = "female"

last_name,first_name,purchase_date
Mouse,Minnie,2016-10-14
Swabota,Maria,2015-12-01
Korkova,Maria,2022-12-12
Mouse,Minnie,2020-08-14


9. On veut connaitre, pour tous les clients nommés Parker, la marque et le modèle des voitures qu'ils ont acheté.

Afficher la liste des noms et prénoms de ces clients ainsi que la marque et le modèle de leurs voitures.

In [17]:
/* Écrire votre requête
On doit trouver une liste de 4 personnes */
SELECT last_name, first_name, brand_name, model_name
FROM customers cu, ownerships o, cars ca, models m, brands b
WHERE cu.customer_id = o.customer_id
AND o.vin = ca.vin
AND ca.model_id = m.model_id
AND m.brand_id = b.brand_id
AND LOWER(last_name) = "parker"

last_name,first_name,brand_name,model_name
Parker,Jessica,Yellow,Orange
Parker,Pamela,Ferrari,450
Parker,Tony,Yellow,Blue
Parker,Tony,Ferrari,F40


10. Afficher la somme des prix d'achat (`purchase_price`) de toutes les voitures que la marque (`brand_name`) _Ferrari_ a vendu.

In [ ]:
/* Écrire votre requête
On doit trouver 978700 dollars */


11. Afficher, sans doublon,  la liste des modèles de voitures équipées de 4 roues motrices (elles ont un chassis dont le `part_name` vaut `'4WD Chassis'`), ainsi que leur marque.

#### D - Modification de la base

12. La voiture n°7 existe bien : en effet, la requête suivante renvoie un résultat non vide.

In [ ]:
SELECT cars.vin, model_name
FROM cars
JOIN models ON models.model_id = cars.model_id
WHERE cars.vin = 7;

Mais elle n'a pas encore été vendue : en effet, la requête suivante renvoie un résultat vide.

In [ ]:
SELECT cars.vin, customer_id
FROM cars
JOIN ownerships ON ownerships.vin = cars.vin
WHERE cars.vin = 7;

Écrire la requête permettant de matérialiser l'achat de cette voiture par la cliente Maria Korkova (_il faudra peut-être écrire au préalable une requête permettant de trouver l'identifiant de cette cliente_), avec les caractéristiques suivantes :

- `purchase_date` : la date d'aujourd'hui, au format `"AAAA-MM-DD"` ;
- `purchase_price` : 26200 dollars
- `warantee_expire_date` : 5 ans de plus que la date d'aujourd'hui, également au format `"AAAA-MM-DD"`.

Vérifier ensuite par une requête que Maria Korkova est bien la propriétaire de cette voiture.

In [ ]:
/* Écrire ici la requête
Maria Korkova doit maintenant être prorriétaire de deux voitures */

13. Le client Lancelot DuLac vient d'avoir une augmentation de salaire : son `household_income` est maintenant de 250000 dollars. Écrire la requête qui permet de mettre à jour la base.